# OCI Specialist - Kaggle 8B

In [ ]:
#@title Install
!pip install -q peft transformers accelerate datasets
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
print(MODEL_NAME)

In [ ]:
#@title Dataset
import json
DATA_PATH = "/kaggle/input/oci-llm-training/"
def load(path):
    d = []
    with open(path) as f:
        for line in f: d.append(json.loads(line))
    return d
train_d = load(DATA_PATH + "train.jsonl")
valid_d = load(DATA_PATH + "valid.jsonl")
print(len(train_d), len(valid_d))

In [ ]:
#@title Format & Tokenize
from transformers import AutoTokenizer
from datasets import Dataset
def fmt(x):
    s = u = a = ""
    for m in x['messages']:
        if m['role']=='system': s=m['content']
        elif m['role']=='user': u=m['content']
        elif m['role']=='assistant': a=m['content']
    return {'text': f"<|start_header_id|>system<|end_header_id|>{s}<|eot_id|><|start_header_id|>user<|end_header_id|>{u}<|eot_id|><|start_header_id|>assistant<|end_header_id|>{a}<|eot_id|>"}
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
def tk(ex): return tok(ex['text'], truncation=True, max_length=2048, padding='max_length')
train_ds = Dataset.from_list([fmt(d) for d in train_d])
valid_ds = Dataset.from_list([fmt(d) for d in valid_d])
train_ds = train_ds.map(tk, batched=True).remove_columns('text')
valid_ds = valid_ds.map(tk, batched=True).remove_columns('text')
print(len(train_ds), len(valid_ds))

In [ ]:
#@title Load Model
import torch
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map='auto', torch_dtype=torch.float16)
print("loaded")

In [ ]:
#@title LoRA
from peft import LoraConfig, get_peft_model, TaskType
lc = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj","v_proj"], lora_dropout=0, task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lc)
model.print_trainable_parameters()

In [ ]:
#@title Train
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
args = TrainingArguments(output_dir="./out", num_train_epochs=1, per_device_train_batch_size=1, learning_rate=2e-5, fp16=True, save_steps=500, eval_steps=500, logging_steps=50, gradient_accumulation_steps=4, load_best_model_at_end=True)
tr = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds, data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
tr.train()

In [ ]:
#@title Save
model.save_adapter("./ad", "default")
import zipfile
with zipfile.ZipFile('adapter.zip','w') as z:
    z.write('./ad/adapter_config.json')
    z.write('./ad/adapter_model.safetensors')
print("done")